# 16 — Embeddings and cosine similarity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/016-embeddings.ipynb)

A word becomes a point, and closeness becomes meaning. In this notebook you place four words on a two-dimensional plane, measure the angle between every pair with cosine, and spin one vector to see the score move. Finally you watch the raw dot product get fooled by length.

## 1. The numbers: four words as points

Each word is a pair of numbers, so it can be drawn on paper. Two dimensions are a toy, but the arithmetic is exactly what a 768-dimensional model does. Notice that the lengths already differ, which matters later.

In [ ]:
import numpy as np                              # the array library, preinstalled

emb = {"cat": np.array([0.9, 0.8]),             # an animal, upper right
       "kitten": np.array([0.85, 0.9]),         # a young animal, almost the same spot
       "dog": np.array([0.7, 0.1]),             # an animal, but a different corner
       "car": np.array([-0.8, 0.3])}            # a vehicle, far to the left

for w, v in emb.items():                        # one line per word
    print(f"{w:>7}: {v}  length = {np.linalg.norm(v):.2f}")   # vector and its length

**Line by line**

- `import numpy as np` — NumPy, already installed on Colab.
- `emb = {...}` — a dictionary from word to its vector.
- `np.array([0.9, 0.8])` — two numbers: the word's coordinates.
- `np.linalg.norm(v)` — the length of the arrow from the origin.
- `for w, v in emb.items()` — walk the dictionary, name and vector.

## 2. Step by step: fill the cosine table

Cosine divides the dot product by both lengths, so only direction is left. The result runs from 1 for identical directions through 0 for perpendicular to -1 for opposite. Fill a table cell by cell with a double loop.

In [ ]:
names = list(emb)                              # the words in a fixed order

def cos(a, b):                                  # cosine of the angle between two vectors
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))   # dot divided by both lengths

M = np.zeros((len(names), len(names)))          # an empty similarity table
for i, a in enumerate(names):                   # row: the first word
    for j, b in enumerate(names):               # column: the second word
        M[i, j] = cos(emb[a], emb[b])           # fill one cell
for i, a in enumerate(names):                   # print row by row
    row = " ".join(f"{M[i, j]:+.2f}" for j in range(len(names)))   # one formatted row
    print(f"{a:>7} {row}")                      # word label, then its similarities
print("diagonal all ones:", np.allclose(np.diag(M), 1))   # a vector matches itself exactly

**Line by line**

- `names = list(emb)` — fixes an order so rows and columns agree.
- `def cos(a, b)` — a helper used everywhere below.
- `(a @ b) / (norm(a) * norm(b))` — the dot product with both lengths removed.
- `M = np.zeros((n, n))` — the table, n rows by n columns.
- `M[i, j] = cos(...)` — one cell per pair, including the diagonal.
- `f"{M[i, j]:+.2f}"` — always print a sign, then two decimals.
- `np.diag(M)` — the diagonal of a matrix.

## 3. Plot: the words on a plane

The picture is the table seen from above. Points close to each other share a direction, and the two animal words crowd into the upper right while the vehicle sits alone. Direction, not distance to the origin, is what cosine reads.

In [ ]:
import matplotlib.pyplot as plt                 # plotting library, preinstalled

plt.figure(figsize=(5.5, 4))                    # near-square figure for a plane
for w, v in emb.items():                        # one labelled dot per word
    plt.scatter(v[0], v[1], s=90)               # dot at the vector's tip
    plt.annotate(w, v, xytext=(6, 6), textcoords="offset points")   # name beside the dot
    plt.plot([0, v[0]], [0, v[1]], color="grey", linewidth=0.7)     # arrow line to the origin
plt.axhline(0, color="grey", linewidth=0.6)     # x-axis
plt.axvline(0, color="grey", linewidth=0.6)     # y-axis
plt.xlim(-1.3, 1.3)                             # keep the origin centred
plt.ylim(-0.3, 1.3)                             # some empty space above
plt.xlabel("dimension 1 (roughly animal-ness)")  # x-axis label
plt.ylabel("dimension 2 (roughly size)")        # y-axis label
plt.title("cat and kitten point the same way; car points back")   # the takeaway
plt.show()                                      # render the figure

**Line by line**

- `plt.figure(figsize=...)` — set the shape before drawing.
- `plt.scatter(x, y, s=90)` — a dot with size 90.
- `plt.annotate(...)` — the word text, offset a few points from the dot.
- `plt.plot([0, v[0]], [0, v[1]])` — a line from the origin to the tip.
- `plt.axhline` / `plt.axvline` — the two axes through zero.
- `plt.xlim` / `plt.ylim` — fixed ranges so the aspect is honest.

## 4. Interactive: rotate the car

The slider rotates the `car` vector around the origin, which leaves its length untouched and changes only its direction. Cosine with `cat` sweeps from negative through zero to positive as the arrow turns. Watch the title: it prints the live score.

In [ ]:
from ipywidgets import interact                   # sliders that rerun a function

def rotate(angle=0):                            # angle in degrees, from the slider
    th = np.deg2rad(angle)                      # degrees to radians
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])   # rotation matrix
    car = R @ emb["car"]                        # the car, rotated
    s = cos(emb["cat"], car)                    # cosine with the fixed cat vector
    plt.figure(figsize=(5, 4))                  # fresh figure each drag
    for w, v in emb.items():                    # draw the fixed words
        if w != "car":                          # car is drawn separately
            plt.scatter(v[0], v[1], s=70, color="tab:grey")   # fixed point
            plt.annotate(w, v, xytext=(6, 6), textcoords="offset points")   # its name
    plt.plot([0, car[0]], [0, car[1]], color="tab:red", linewidth=1.5)   # rotating arrow shaft
    plt.scatter(car[0], car[1], s=90, color="tab:red")   # rotating tip
    plt.annotate("car", car, xytext=(6, 6), textcoords="offset points")   # label it
    plt.axhline(0, color="grey", linewidth=0.6)  # x-axis
    plt.axvline(0, color="grey", linewidth=0.6)  # y-axis
    plt.xlim(-1.5, 1.5)                          # fixed limits, nothing jumps
    plt.ylim(-1.5, 1.5)                          # same on y
    plt.title(f"{angle} deg -> cos(cat, car) = {s:+.2f}")   # live readout
    plt.show()                                  # draw the frame

interact(rotate, angle=(0, 360, 10))            # slider: 0 to 360 degrees

**Line by line**

- `interact` — the widget that calls the function again.
- `def rotate(angle=0)` — one frame per slider position.
- `R = np.array([...])` — the 2-by-2 matrix that rotates a vector.
- `car = R @ emb["car"]` — matrix times vector gives the turned vector.
- `s = cos(...)` — recompute the score for the readout.
- `plt.axhline` / `plt.xlim` — fixed axes so the arrow turns in place.
- `interact(..., angle=(0, 360, 10))` — range and step of the slider.

## 5. When it breaks: the dot product counts length

Repeat a word and the dot product grows even though the meaning never changed. Here a document of eight dogs outscores the single real cat, purely because it is longer. Cosine puts the two cat documents back on top.

In [ ]:
docs = {"cat": emb["cat"],                    # the query, one cat
        "cat x8": 8 * emb["cat"],               # same meaning, eight times longer
        "dog x8": 8 * emb["dog"],               # different meaning, eight times longer
        "car": emb["car"]}                      # unrelated
q = emb["cat"]                                  # the query vector
dots = {k: q @ v for k, v in docs.items()}      # raw dot: grows with length
coss = {k: cos(q, v) for k, v in docs.items()}  # cosine: length removed
for k in docs:                                  # print both scores
    print(f"{k:<7} dot={dots[k]:6.2f}  cos={coss[k]:+.2f}")   # side by side

pos = np.arange(len(docs))                      # one x per document
plt.figure(figsize=(6.5, 3))                    # wide, short figure
plt.bar(pos - 0.2, [dots[k] for k in docs], width=0.4, color="tab:orange", label="dot")   # fooled by length
plt.bar(pos + 0.2, [coss[k] for k in docs], width=0.4, color="tab:green", label="cosine")  # only direction
plt.xticks(pos, list(docs))                     # document names
plt.axhline(0, color="black", linewidth=0.6)    # zero line
plt.ylabel("score against the query")           # y-axis
plt.title("'dog x8' beats the real cat on dot; cosine fixes it")   # the takeaway
plt.legend()                                    # name the two colours
plt.show()                                      # render the figure

**Line by line**

- `docs = {...}` — four documents, two of them padded by repetition.
- `8 * emb["dog"]` — same direction, eight times the length.
- `dots = {...}` — raw dot product for every document.
- `coss = {...}` — the same pairs under cosine.
- `plt.bar(pos - 0.2, ...)` — raw dot bars, shifted left of the tick.
- `plt.bar(pos + 0.2, ...)` — cosine bars, shifted right.

## Your turn

Drag the section 4 slider to 180 and read the title before checking the table. Then change `"dog x8"` in section 5 to `30 * emb["dog"]` and predict, before rerunning, whether it beats the single cat on the dot product and on cosine.